# Countdown reinforcement fine-tuning

This notebook compares two reinforcement fine-tuning approaches for the same Countdown task:

1. a `gpt-5.4-mini` model/score grader;
2. a deterministic Python grader.

Both jobs use the checked-in 100-row training and 50-row validation datasets, `Alibaba/qwen3.6-35b-a3b` version `1`, and `GlobalStandard` training. The separate 100-row evaluation file is validated for dataset provenance but is not uploaded for training.

The notebook validates the datasets and graders, configures the model-grader deployment, submits or resumes both jobs, monitors them to terminal state, and summarizes the service validation-reward curves.

> Training and grader deployment capacity can incur charges. Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Configure the project and grader deployment

Copy this cookbook's `.env.template` to a module-local `.env`. Configure a Foundry project that supports the training model and a `gpt-5.4-mini` `DataZoneStandard` deployment in the same Azure AI Services account. The score grader submits the configured `FOUNDRY_GRADER_DEPLOYMENT` name; its underlying model/version/SKU are independently verified.

The Azure CLI performs the ARM quota and deployment operations using the identity selected by `az login`. `DefaultAzureCredential` authenticates the Foundry SDK operations and may select the same Azure CLI identity or another configured credential source. Both identities need access to the configured subscription, resource group, account, and project.

By default, `FOUNDRY_GRADER_CAPACITY=max` assigns the grader deployment all currently available matching quota. You may instead set an explicit capacity-unit limit without editing the notebook. The notebook requires the resulting deployment rate limit to be at least `FOUNDRY_GRADER_MIN_TPM` (default `100000` TPM) so the score-model sampling evaluation has sufficient throughput. Review the printed read-only plan, then set `FOUNDRY_ALLOW_GRADER_DEPLOYMENT_UPDATE=true` to authorize creation or update.

Before the first ARM mutation, `outputs/grader-deployment-state.json` saves an immutable operation intent: the configuration/recipe identity, whether the deployment existed, its previous resource ID and capacity, and the first requested capacity. Reruns validate that record and the live deployment without resizing, even if no model-grader job was recorded yet. Failed or ambiguous updates retain the first cleanup baseline; a missing or unexpectedly sized deployment requires manual reconciliation rather than another automatic update. A recorded model-grader job or pending submission is also verification-only. Legacy or mismatched state fails closed.

Executing the deployment, upload, and submission cells performs billable operations and submits or resumes two training jobs. Reruns reuse the recorded job IDs only when the recipe fingerprint, including the configured grader deployment and stable grader names, matches. Per-label pending submission intents are saved before POST and never cleared to authorize a retry: an unknown outcome without a known ID requires independent operator reconciliation. The notebook does not automatically restore or delete deployments; the operation-intent record is not proof of ownership or successful creation.

In [ ]:
import ast
import csv
import hashlib
import io
import json
import math
import operator
import os
import shutil
import subprocess
import tempfile
import time
from collections import Counter
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def find_demo_root():
    current = Path.cwd().resolve()
    candidates = [current, current.parent] if current.name == "notebooks" else [current]
    for candidate in candidates:
        if (
            (candidate / "data" / "preserved" / "countdown-training-rft-100.jsonl").exists()
            and (candidate / "notebooks" / "demo.ipynb").exists()
        ):
            return candidate
    raise FileNotFoundError("Run this notebook from the Countdown cookbook or its notebooks directory.")


def env_bool(name, default=False):
    value = os.getenv(name, "").strip().casefold() or str(default).casefold()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be true or false.")
    return value == "true"


def positive_integer(name, default):
    value = os.getenv(name, "").strip() or str(default)
    if not value.isdecimal() or int(value) < 1:
        raise ValueError(f"{name} must be a positive integer.")
    return int(value)


DEMO_ROOT = find_demo_root()
OUTPUT_ROOT = DEMO_ROOT / "outputs"
STATE_PATH = OUTPUT_ROOT / "submission-state.json"
GRADER_DEPLOYMENT_STATE_PATH = OUTPUT_ROOT / "grader-deployment-state.json"


from dotenv import load_dotenv

load_dotenv(DEMO_ROOT / ".env", override=False)


PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()
SUBSCRIPTION_ID = os.getenv("FOUNDRY_SUBSCRIPTION_ID", "").strip()
RESOURCE_GROUP = os.getenv("FOUNDRY_RESOURCE_GROUP", "").strip()
RESOURCE_LOCATION = os.getenv("FOUNDRY_RESOURCE_LOCATION", "").strip()

MODEL = "qwen3.6-35b-a3b"
MODEL_PUBLISHER = "Alibaba"
MODEL_VERSION = "1"
TRAINING_TYPE = "globalStandard"
GRADER_MODEL = "gpt-5.4-mini"
GRADER_MODEL_VERSION = "2026-03-17"
GRADER_DEPLOYMENT = os.getenv("FOUNDRY_GRADER_DEPLOYMENT", "").strip() or GRADER_MODEL
GRADER_DEPLOYMENT_SKU = "DataZoneStandard"
GRADER_QUOTA_NAME = "OpenAI.DataZoneStandard.gpt-5.4-mini"
GRADER_CAPACITY_SETTING = os.getenv("FOUNDRY_GRADER_CAPACITY", "").strip().casefold() or "max"
GRADER_MIN_TPM = positive_integer("FOUNDRY_GRADER_MIN_TPM", 100_000)
ALLOW_GRADER_DEPLOYMENT_UPDATE = env_bool(
    "FOUNDRY_ALLOW_GRADER_DEPLOYMENT_UPDATE",
    False,
)


print("Project:", PROJECT_ENDPOINT)
print("Region:", RESOURCE_LOCATION)
print("Model:", f"{MODEL_PUBLISHER}/{MODEL}, version {MODEL_VERSION}")
print("Training type:", TRAINING_TYPE)
print("Grader deployment:", GRADER_DEPLOYMENT, GRADER_MODEL, GRADER_DEPLOYMENT_SKU)
print("Grader capacity:", GRADER_CAPACITY_SETTING, f"(minimum {GRADER_MIN_TPM:,} TPM)")

## 2. Validate the exact datasets

In [ ]:
DATASETS = {
    "training": {
        "path": DEMO_ROOT / "data" / "preserved" / "countdown-training-rft-100.jsonl",
        "bytes": 126202,
        "rows": 100,
        "sha256": "5dfdd0dfcbab25119d79ea7e423fde29b22db41db26cc939407c7be41d801307",
    },
    "validation": {
        "path": DEMO_ROOT / "data" / "preserved" / "countdown-validation-rft-50.jsonl",
        "bytes": 63104,
        "rows": 50,
        "sha256": "d6dcfef82e8bf23a7cbd111d1dd9793763f5dd640c3110daba5fa1453638eb9c",
    },
    "evaluation_provenance": {
        "path": DEMO_ROOT / "data" / "preserved" / "countdown-evaluation-100.jsonl",
        "bytes": 5435,
        "rows": 100,
        "sha256": "d433d597e8ebc8f255347be5724591c4b6776bda97fdc5e30ac6ff1d5e4f48dd",
    },
}


def validate_jsonl(label, contract):
    raw_content = contract["path"].read_bytes()
    content = raw_content.replace(b"\r\n", b"\n")
    rows = [json.loads(line) for line in content.splitlines() if line.strip()]
    actual = {
        "raw_bytes": len(raw_content),
        "raw_sha256": hashlib.sha256(raw_content).hexdigest(),
        "normalized_bytes": len(content),
        "rows": len(rows),
        "normalized_sha256": hashlib.sha256(content).hexdigest(),
    }
    normalized = {
        "bytes": actual["normalized_bytes"],
        "rows": actual["rows"],
        "sha256": actual["normalized_sha256"],
    }
    expected = {key: contract[key] for key in ("bytes", "rows", "sha256")}
    if normalized != expected:
        raise ValueError(f"{label} integrity mismatch: expected {expected}, got {actual}")
    for index, row in enumerate(rows, start=1):
        if label == "evaluation_provenance":
            if set(row) != {"item"} or set(row["item"]) != {"target", "nums"}:
                raise ValueError(
                    f"{label} row {index} has unexpected keys: {sorted(row)}"
                )
            item = row["item"]
        else:
            if set(row) != {"messages", "target", "nums"}:
                raise ValueError(
                    f"{label} row {index} has unexpected keys: {sorted(row)}"
                )
            if len(row["messages"]) != 1 or row["messages"][0].get("role") != "user":
                raise ValueError(f"{label} row {index} must contain one user message")
            item = row
        numbers = json.loads(item["nums"])
        if not isinstance(numbers, list) or not numbers:
            raise ValueError(f"{label} row {index} has invalid nums")
        float(item["target"])
    return rows, actual


validated_datasets = {}
for label, contract in DATASETS.items():
    rows, evidence = validate_jsonl(label, contract)
    validated_datasets[label] = rows
    print(label, evidence)

## 3. Validate the shared response and grader contracts

In [ ]:
RESPONSE_FORMAT = json.loads(
    (DEMO_ROOT / "graders" / "preserved" / "countdown-response-schema.json").read_text(
        encoding="utf-8"
    )
)

MODEL_GRADER = {
    "name": "countdown_model_grader",
    "type": "score_model",
    "model": GRADER_DEPLOYMENT,
    "input": [
        {
            "role": "developer",
            "content": (
                "Grade a Countdown answer. Verify that the JSON output has expression and result, "
                "that the expression uses each input number exactly once, uses only +, -, *, / and "
                "parentheses, that the reported result equals the expression, and that it reaches "
                "the target. Score 5 for an exact valid solution, 4 when off by 1, 3 when off by "
                "2-5, 2 when farther away but otherwise valid, 1 for a result mismatch, and 0 for "
                "invalid syntax or number usage. Return only a score and brief reasoning."
            ),
        },
        {
            "role": "user",
            "content": (
                "target={{item.target}} numbers={{item.nums}} "
                "output={{sample.output_text}}"
            ),
        },
    ],
    "pass_threshold": 5,
    "range": [0.0, 5.0],
}

PYTHON_GRADER_SOURCE = r'''import ast
import json
import math
import operator
from collections import Counter

_MAX_EXPRESSION_CHARS = 4096
_MAX_AST_NODES = 256
_MAX_AST_DEPTH = 64

_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _parse_expression(expression):
    if not isinstance(expression, str) or not 0 < len(expression) <= _MAX_EXPRESSION_CHARS:
        raise ValueError("Expression length is outside the supported limit")
    tree = ast.parse(expression, mode="eval")
    pending = [(tree, 1)]
    count = 0
    while pending:
        node, depth = pending.pop()
        count += 1
        if count > _MAX_AST_NODES or depth > _MAX_AST_DEPTH:
            raise ValueError("Expression AST is outside the supported complexity limit")
        pending.extend((child, depth + 1) for child in ast.iter_child_nodes(node))
    return tree


def _evaluate(node, numbers):
    if isinstance(node, ast.Expression):
        return _evaluate(node.body, numbers)
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        value = float(node.value)
        if not math.isfinite(value):
            raise ValueError("Numeric literals must be finite")
        numbers.append(value)
        return value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPERATORS:
        value = _OPERATORS[type(node.op)](
            _evaluate(node.left, numbers),
            _evaluate(node.right, numbers),
        )
        if not math.isfinite(value):
            raise ValueError("Calculated values must be finite")
        return value
    raise ValueError("Only numeric literals and +, -, *, / are allowed")


def grade(sample, item) -> float:
    try:
        output = sample.get("output_json")
        if output is None:
            output = json.loads(sample["output_text"])
        expression = output["expression"]
        used_numbers = []
        calculated = _evaluate(_parse_expression(expression), used_numbers)
        input_numbers = [float(value) for value in json.loads(item["nums"])]
        reported = float(output["result"])
        target = float(item["target"])
        if not all(math.isfinite(value) for value in (
            calculated, reported, target, *used_numbers, *input_numbers,
        )):
            return 0.0
        if Counter(used_numbers) != Counter(input_numbers):
            return 0.0
        result_difference = abs(calculated - reported)
        difference = abs(calculated - target)
        if not math.isfinite(result_difference) or not math.isfinite(difference):
            return 0.0
        if result_difference > 1e-9:
            return 1.0
        if difference < 1e-9:
            return 5.0
        if difference <= 1:
            return 4.0
        if difference <= 5:
            return 3.0
        return 2.0
    except (
        KeyError,
        TypeError,
        ValueError,
        SyntaxError,
        ZeroDivisionError,
        OverflowError,
        RecursionError,
        json.JSONDecodeError,
    ):
        return 0.0
'''

PYTHON_GRADER = {
    "name": "countdown_python_grader",
    "type": "python",
    "source": PYTHON_GRADER_SOURCE,
}

HYPERPARAMETERS = {
    "eval_interval": 5,
    "eval_samples": 2,
    "reasoning_effort": "medium",
    "n_epochs": 1,
    "batch_size": 4,
    "learning_rate_multiplier": 2,
}


def validate_grader_contracts():
    if MODEL_GRADER.get("name") != "countdown_model_grader":
        raise ValueError("Model grader name changed")
    if MODEL_GRADER.get("type") != "score_model" or MODEL_GRADER.get("range") != [0.0, 5.0]:
        raise ValueError("Model grader contract changed")
    if MODEL_GRADER.get("pass_threshold") != 5 or MODEL_GRADER.get("model") != GRADER_DEPLOYMENT:
        raise ValueError("Model grader threshold or configured deployment changed")
    if PYTHON_GRADER.get("name") != "countdown_python_grader":
        raise ValueError("Python grader requires the stable countdown_python_grader name")
    if PYTHON_GRADER.get("type") != "python" or PYTHON_GRADER.get("source") != PYTHON_GRADER_SOURCE:
        raise ValueError("Python grader type or source changed")
    if RESPONSE_FORMAT.get("type") != "json_schema":
        raise ValueError("Expected a JSON-schema response format")


def build_job_payload(label, grader, suffix, training_file, validation_file):
    # Validate both variants even when the model-grader job is submitted first.
    validate_grader_contracts()
    expected_grader = {
        "model-grader": MODEL_GRADER,
        "python-grader": PYTHON_GRADER,
    }.get(label)
    if expected_grader is None or grader != expected_grader:
        raise ValueError("The job label does not match its validated grader")
    return {
        "model": MODEL,
        "training_file": training_file,
        "validation_file": validation_file,
        "method": {
            "type": "reinforcement",
            "reinforcement": {
                "hyperparameters": HYPERPARAMETERS,
                "grader": grader,
                "response_format": RESPONSE_FORMAT,
            },
        },
        "suffix": suffix,
        "extra_body": {"trainingType": TRAINING_TYPE},
    }


def recipe_fingerprint():
    validate_grader_contracts()
    return hashlib.sha256(
        json.dumps(
            {
                "endpoint": PROJECT_ENDPOINT,
                "model": MODEL,
                "model_version": MODEL_VERSION,
                "training_type": TRAINING_TYPE,
                "training_sha256": DATASETS["training"]["sha256"],
                "validation_sha256": DATASETS["validation"]["sha256"],
                "hyperparameters": HYPERPARAMETERS,
                "grader_deployment": {
                    "name": GRADER_DEPLOYMENT,
                    "model": GRADER_MODEL,
                    "model_version": GRADER_MODEL_VERSION,
                    "sku": GRADER_DEPLOYMENT_SKU,
                },
                "model_grader": MODEL_GRADER,
                "python_grader": PYTHON_GRADER,
                "response_format": RESPONSE_FORMAT,
            },
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()


RUN_FINGERPRINT = recipe_fingerprint()


def canonical_json_sha256(value):
    content = json.dumps(
        value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(content).hexdigest()


def submission_reconciliation_message(label=None, job_id=None):
    scope = f" for {label}" if label is not None else ""
    returned_id = f" Returned job ID: {job_id!r}." if job_id is not None else ""
    return (
        f"Operator reconciliation is required{scope}.{returned_id} "
        "Keep the original submission-state.json bytes and pending evidence. "
        "Never clear, delete, or archive pending to resubmit. Independently verify the "
        "Foundry project and match the saved endpoint, payload/suffix, exact file IDs/hashes, "
        "and service request/audit evidence. Record only an independently verified matching "
        "job ID in jobs for this label, retaining pending, then rerun to retrieve it. "
        "If no matching ID can be proved, stop and contact the service operator; absence "
        "from a job listing is not proof that no job was accepted."
    )


def submission_file_evidence(payload):
    return {
        kind: {
            "file_id": payload[f"{kind}_file"],
            **{key: DATASETS[kind][key] for key in ("bytes", "rows", "sha256")},
        }
        for kind in ("training", "validation")
    }


def validate_submission_intent(label, intent):
    message = "Invalid or unverified pending submission intent. " + submission_reconciliation_message(label)
    required = {
        "schema_version", "record_type", "recorded_at", "label", "project_endpoint",
        "run_fingerprint", "payload", "payload_sha256", "files",
    }
    if not isinstance(intent, dict) or set(intent) != required:
        raise RuntimeError(message)
    if (
        type(intent["schema_version"]) is not int
        or intent["schema_version"] != 1
        or intent["record_type"] != "job_creation_intent"
        or intent["label"] != label
        or intent["project_endpoint"] != PROJECT_ENDPOINT
        or intent["run_fingerprint"] != RUN_FINGERPRINT
    ):
        raise RuntimeError(message)
    try:
        recorded_at = datetime.fromisoformat(intent["recorded_at"])
        if recorded_at.tzinfo is None or recorded_at.utcoffset() != timezone.utc.utcoffset(None):
            raise ValueError("The intent needs an explicit UTC timestamp")
        payload = intent["payload"]
        if not isinstance(payload, dict):
            raise ValueError("The intent needs its exact payload")
        for key in ("training_file", "validation_file", "suffix"):
            value = payload[key]
            if not isinstance(value, str) or not value or value != value.strip():
                raise ValueError("The intent needs exact nonempty file IDs and suffix")
        expected_grader = {"model-grader": MODEL_GRADER, "python-grader": PYTHON_GRADER}[label]
        expected_payload = build_job_payload(
            label, expected_grader, payload["suffix"],
            payload["training_file"], payload["validation_file"],
        )
        actual_hash = canonical_json_sha256(payload)
        if (
            actual_hash != canonical_json_sha256(expected_payload)
            or intent["payload_sha256"] != actual_hash
            or canonical_json_sha256(intent["files"]) != canonical_json_sha256(
                submission_file_evidence(payload)
            )
        ):
            raise ValueError("The pending payload or file evidence does not match the recipe")
    except (KeyError, TypeError, ValueError, OverflowError) as exc:
        raise RuntimeError(message) from exc


def build_submission_intent(label, payload):
    # Detach the durable evidence from mutable caller/SDK dictionaries.
    payload = json.loads(json.dumps(payload, allow_nan=False))
    for kind in ("training", "validation"):
        validate_jsonl(kind, DATASETS[kind])
    intent = {
        "schema_version": 1,
        "record_type": "job_creation_intent",
        "recorded_at": datetime.now(timezone.utc).isoformat(),
        "label": label,
        "project_endpoint": PROJECT_ENDPOINT,
        "run_fingerprint": RUN_FINGERPRINT,
        "payload": payload,
        "payload_sha256": canonical_json_sha256(payload),
        "files": submission_file_evidence(payload),
    }
    validate_submission_intent(label, intent)
    return intent


def load_state():
    if not STATE_PATH.exists():
        return {"schema_version": 2, "fingerprint": None, "jobs": {}, "pending": {}}
    message = "Invalid submission state. " + submission_reconciliation_message()
    try:
        state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise RuntimeError(message) from exc
    allowed = {"schema_version", "fingerprint", "jobs", "pending", "job_endpoints"}
    if (
        not isinstance(state, dict)
        or set(state) - allowed
        or not isinstance(state.get("jobs"), dict)
    ):
        raise RuntimeError(message)
    if "schema_version" not in state:
        # Existing job-only records remain readable, but never infer pending evidence.
        if "pending" in state:
            raise RuntimeError(message)
        pending = {}
    elif (
        type(state["schema_version"]) is not int
        or state["schema_version"] != 2
        or not isinstance(state.get("pending"), dict)
    ):
        raise RuntimeError(message)
    else:
        pending = state["pending"]
    labels = {"model-grader", "python-grader"}
    if set(state["jobs"]) - labels or set(pending) - labels:
        raise RuntimeError(message)
    for job_id in state["jobs"].values():
        if not isinstance(job_id, str) or not job_id or job_id != job_id.strip():
            raise RuntimeError(message)
    fingerprint = state.get("fingerprint")
    if (fingerprint is not None and fingerprint != RUN_FINGERPRINT) or (
        (state["jobs"] or pending) and fingerprint is None
    ):
        raise RuntimeError(
            "The existing output state belongs to a different or unverified recipe. "
            + submission_reconciliation_message()
        )
    if "job_endpoints" in state:
        endpoints = state["job_endpoints"]
        if (
            not isinstance(endpoints, dict)
            or set(endpoints) - set(state["jobs"])
            or any(endpoint != PROJECT_ENDPOINT for endpoint in endpoints.values())
        ):
            raise RuntimeError(message)
    for label, intent in pending.items():
        validate_submission_intent(label, intent)
    # Normalize only in memory. Resume/refusal never rewrites legacy or pending bytes.
    return {**state, "schema_version": 2, "pending": pending}


namespace = {}
exec(PYTHON_GRADER_SOURCE, namespace)
grade = namespace["grade"]
grader_tests = [
    (
        "exact",
        {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "24"}},
        {"target": "24", "nums": "[1, 2, 3, 4]"},
        5,
    ),
    (
        "result mismatch",
        {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "25"}},
        {"target": "24", "nums": "[1, 2, 3, 4]"},
        1,
    ),
    (
        "reused number",
        {"output_json": {"expression": "4 * 4 + 3 + 2", "result": "21"}},
        {"target": "21", "nums": "[1, 2, 3, 4]"},
        0,
    ),
    (
        "unsupported syntax",
        {"output_json": {"expression": "sum([1, 2, 3, 4])", "result": "10"}},
        {"target": "10", "nums": "[1, 2, 3, 4]"},
        0,
    ),
]
for nonfinite in ("NaN", "Infinity", "-Infinity", "1e309"):
    grader_tests.extend([
        (
            f"nonfinite reported result: {nonfinite}",
            {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": nonfinite}},
            {"target": "24", "nums": "[1, 2, 3, 4]"},
            0,
        ),
        (
            f"nonfinite target: {nonfinite}",
            {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "24"}},
            {"target": nonfinite, "nums": "[1, 2, 3, 4]"},
            0,
        ),
        (
            f"nonfinite input number: {nonfinite}",
            {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "24"}},
            {"target": "24", "nums": json.dumps([1, 2, 3, nonfinite])},
            0,
        ),
    ])
grader_tests.extend([
    (
        "nonfinite numeric literal",
        {"output_json": {"expression": "1e309 + 1", "result": "1"}},
        {"target": "1", "nums": "[1e309, 1]"},
        0,
    ),
    (
        "overflowed intermediate",
        {"output_json": {"expression": "(1e308 * 1e308) / 1e308", "result": "1e308"}},
        {"target": "1e308", "nums": "[1e308, 1e308, 1e308]"},
        0,
    ),
    (
        "overflowed target difference",
        {"output_json": {"expression": "1e308 + 0", "result": "1e308"}},
        {"target": "-1e308", "nums": "[1e308, 0]"},
        0,
    ),
    (
        "deep arithmetic chain",
        {"output_json": {"expression": "+".join(["1"] * 1500), "result": "1500"}},
        {"target": "1500", "nums": json.dumps([1] * 1500)},
        0,
    ),
])
for name, sample, item, expected in grader_tests:
    actual = grade(sample, item)
    if actual != expected:
        raise AssertionError(f"{name}: expected {expected}, got {actual}")
    print(name, actual)

## 4. Check or update the grader deployment, then upload the exact 100/50 training inputs

In [ ]:
def run_az_json(*arguments):
    executable = shutil.which("az")
    if not executable:
        raise RuntimeError("Azure CLI is required. Install it and run az login.")
    completed = subprocess.run(
        [executable, *arguments, "--only-show-errors", "-o", "json"],
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip())
    return json.loads(completed.stdout)


def normalized_location(value):
    return "".join(character for character in value.casefold() if character.isalnum())


def validate_existing_grader(deployment):
    if not deployment:
        return
    if deployment.get("name") != GRADER_DEPLOYMENT:
        raise RuntimeError("The ARM response does not match the configured grader deployment name.")
    model = deployment.get("properties", {}).get("model", {})
    expected = {
        "format": "OpenAI",
        "name": GRADER_MODEL,
        "version": GRADER_MODEL_VERSION,
    }
    actual_sku = deployment.get("sku", {}).get("name")
    if any(model.get(key) != value for key, value in expected.items()):
        raise RuntimeError(
            f"Deployment name collision: {GRADER_DEPLOYMENT} serves "
            f"{model.get('name')} {model.get('version')} ({model.get('format')}). "
            "Choose another FOUNDRY_GRADER_DEPLOYMENT; no resource was changed."
        )
    if actual_sku != GRADER_DEPLOYMENT_SKU:
        raise RuntimeError(
            f"Deployment name collision: {GRADER_DEPLOYMENT} uses SKU {actual_sku}. "
            f"Expected {GRADER_DEPLOYMENT_SKU}; no resource was changed."
        )


def choose_grader_capacity(quota_limit, quota_used, existing_capacity, setting):
    maximum = int(quota_limit) - int(quota_used) + int(existing_capacity)
    minimum_capacity = math.ceil(GRADER_MIN_TPM / 1_000)
    if maximum < minimum_capacity:
        raise RuntimeError(
            f"Available grader capacity is {maximum}, below the {minimum_capacity} "
            f"capacity units required for {GRADER_MIN_TPM:,} TPM."
        )
    if setting == "max":
        return maximum
    if not setting.isdecimal() or int(setting) < minimum_capacity:
        raise ValueError(
            "FOUNDRY_GRADER_CAPACITY must be 'max' or an integer of at least "
            f"{minimum_capacity} capacity units."
        )
    requested = int(setting)
    if requested > maximum:
        raise RuntimeError(
            f"Requested grader capacity {requested} exceeds the available maximum {maximum}."
        )
    return requested


def deployment_token_limit(deployment):
    for rate_limit in deployment.get("properties", {}).get("rateLimits", []):
        if rate_limit.get("key") == "token":
            return int(rate_limit["count"])
    raise RuntimeError("The grader deployment response did not include a token rate limit.")


def grader_account_name():
    hostname = urlparse(PROJECT_ENDPOINT).hostname or ""
    suffix = ".services.ai.azure.com"
    if not hostname.endswith(suffix):
        raise ValueError(
            "FOUNDRY_PROJECT_ENDPOINT must use the account.services.ai.azure.com format."
        )
    return hostname.removesuffix(suffix)


def grader_deployment_identity():
    return {
        "project_endpoint": PROJECT_ENDPOINT,
        "subscription_id": SUBSCRIPTION_ID,
        "resource_group": RESOURCE_GROUP,
        "resource_location": normalized_location(RESOURCE_LOCATION),
        "account_name": grader_account_name(),
        "deployment_name": GRADER_DEPLOYMENT,
        "model": GRADER_MODEL,
        "model_version": GRADER_MODEL_VERSION,
        "sku": GRADER_DEPLOYMENT_SKU,
        "capacity_setting": GRADER_CAPACITY_SETTING,
        "minimum_tpm": GRADER_MIN_TPM,
        "run_fingerprint": RUN_FINGERPRINT,
    }


def load_grader_deployment_state():
    if not GRADER_DEPLOYMENT_STATE_PATH.exists():
        return None
    evidence = json.loads(GRADER_DEPLOYMENT_STATE_PATH.read_text(encoding="utf-8"))
    if not isinstance(evidence, dict) or evidence.get("schema_version") != 1:
        raise RuntimeError(
            "Legacy or invalid grader deployment state; reconcile it before deliberately "
            "archiving it. The original cleanup baseline was not overwritten."
        )
    if any(evidence.get(key) != value for key, value in grader_deployment_identity().items()):
        raise RuntimeError(
            "The saved grader deployment operation belongs to a different configuration "
            "or recipe. Reconcile it before rerunning; the cleanup baseline was not overwritten."
        )
    previous = evidence.get("previous_capacity")
    requested = evidence.get("requested_capacity")
    if (
        type(evidence.get("previously_existed")) is not bool
        or type(previous) is not int
        or previous < 0
        or type(requested) is not int
        or requested < math.ceil(GRADER_MIN_TPM / 1_000)
        or (not evidence["previously_existed"] and previous != 0)
        or (GRADER_CAPACITY_SETTING != "max" and str(requested) != GRADER_CAPACITY_SETTING)
    ):
        raise RuntimeError("Invalid saved deployment baseline; reconcile it without overwriting it.")
    return evidence


def record_grader_deployment_operation(plan, existing):
    evidence = {
        "schema_version": 1,
        "record_type": "operation_intent",
        "recorded_at": datetime.now(timezone.utc).isoformat(),
        **grader_deployment_identity(),
        "previously_existed": existing is not None,
        "previous_deployment_id": (existing or {}).get("id"),
        "previous_capacity": plan["existing_capacity"],
        "requested_capacity": plan["target_capacity"],
    }
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    # Persist the first cleanup baseline BEFORE ARM, including ambiguous/failed updates.
    # Exclusive creation prevents a rerun from replacing it with the resized capacity.
    try:
        with GRADER_DEPLOYMENT_STATE_PATH.open("x", encoding="utf-8") as handle:
            handle.write(json.dumps(evidence, indent=2) + "\n")
            handle.flush()
            os.fsync(handle.fileno())
    except FileExistsError as exc:
        raise RuntimeError(
            "A grader deployment operation is already recorded. Rerun read-only validation "
            "instead of replacing its original cleanup baseline."
        ) from exc


def get_grader_deployment(account_name):
    deployments = run_az_json(
        "cognitiveservices", "account", "deployment", "list",
        "--name", account_name,
        "--resource-group", RESOURCE_GROUP,
        "--subscription", SUBSCRIPTION_ID,
    )
    deployment = next(
        (item for item in deployments if item.get("name") == GRADER_DEPLOYMENT),
        None,
    )
    if deployment:
        deployment = run_az_json(
            "cognitiveservices", "account", "deployment", "show",
            "--name", account_name,
            "--resource-group", RESOURCE_GROUP,
            "--subscription", SUBSCRIPTION_ID,
            "--deployment-name", GRADER_DEPLOYMENT,
        )
    validate_existing_grader(deployment)
    return deployment


def plan_grader_deployment():
    if not all((PROJECT_ENDPOINT, SUBSCRIPTION_ID, RESOURCE_GROUP, RESOURCE_LOCATION)):
        raise RuntimeError(
            "Set FOUNDRY_PROJECT_ENDPOINT, FOUNDRY_SUBSCRIPTION_ID, "
            "FOUNDRY_RESOURCE_GROUP, and FOUNDRY_RESOURCE_LOCATION."
        )
    account_name = grader_account_name()
    account = run_az_json(
        "cognitiveservices", "account", "show",
        "--name", account_name,
        "--resource-group", RESOURCE_GROUP,
        "--subscription", SUBSCRIPTION_ID,
    )
    if account.get("name") != account_name:
        raise RuntimeError("The configured project endpoint does not match the ARM account.")
    if normalized_location(account.get("location", "")) != normalized_location(RESOURCE_LOCATION):
        raise RuntimeError(
            f"FOUNDRY_RESOURCE_LOCATION={RESOURCE_LOCATION} does not match "
            f"the account location {account.get('location')}."
        )

    usage = run_az_json(
        "cognitiveservices", "usage", "list",
        "--location", RESOURCE_LOCATION,
        "--subscription", SUBSCRIPTION_ID,
    )
    quota = next(
        (item for item in usage if item.get("name", {}).get("value") == GRADER_QUOTA_NAME),
        None,
    )
    if not quota:
        raise RuntimeError(f"Quota {GRADER_QUOTA_NAME} is unavailable in {RESOURCE_LOCATION}.")

    existing = get_grader_deployment(account_name)
    existing_capacity = int((existing or {}).get("sku", {}).get("capacity") or 0)
    target_capacity = choose_grader_capacity(
        quota["limit"],
        quota["currentValue"],
        existing_capacity,
        GRADER_CAPACITY_SETTING,
    )
    return {
        "account_name": account_name,
        "deployment_name": GRADER_DEPLOYMENT,
        "model": GRADER_MODEL,
        "model_version": GRADER_MODEL_VERSION,
        "sku": GRADER_DEPLOYMENT_SKU,
        "quota_limit": int(quota["limit"]),
        "quota_used": int(quota["currentValue"]),
        "previously_existed": existing is not None,
        "existing_capacity": existing_capacity,
        "target_capacity": target_capacity,
    }


def ensure_grader_deployment():
    # Check submission/configuration identity before any deployment operation.
    state = load_state()
    evidence = load_grader_deployment_state()
    if recipe_fingerprint() != RUN_FINGERPRINT:
        raise RuntimeError("The recipe changed after validation; rerun the offline preparation cells.")
    plan = plan_grader_deployment()
    account_name = plan["account_name"]
    existing = get_grader_deployment(account_name)

    if (
        evidence is not None
        or state["jobs"].get("model-grader")
        or state["pending"].get("model-grader")
    ):
        if not existing:
            raise RuntimeError(
                "The recorded operation, model-grader job, or pending submission requires the "
                "configured deployment, but it no longer exists. Reconcile it; no resource was changed."
            )
        if evidence is not None and (
            int(existing.get("sku", {}).get("capacity") or 0) != evidence["requested_capacity"]
        ):
            raise RuntimeError(
                "The deployment capacity differs from the first recorded request. Reconcile "
                "the incomplete or externally changed operation; no update was retried "
                "and the original cleanup baseline was preserved."
            )
        token_limit = deployment_token_limit(existing)
        if token_limit < GRADER_MIN_TPM:
            raise RuntimeError(
                f"The recorded grader deployment has {token_limit:,} TPM; "
                f"at least {GRADER_MIN_TPM:,} TPM is required. "
                "Capacity and the original cleanup baseline were not changed during resume."
            )
        print(
            f"Recorded grader operation, job, or pending submission found; verified {GRADER_DEPLOYMENT} "
            f"at {token_limit:,} TPM without changing capacity or the original baseline."
        )
        return existing

    if (
        (existing is not None) != plan["previously_existed"]
        or int((existing or {}).get("sku", {}).get("capacity") or 0) != plan["existing_capacity"]
    ):
        raise RuntimeError("The deployment changed during planning; rerun the read-only checks.")
    print(json.dumps({"grader_deployment_plan": plan}, indent=2))
    if not ALLOW_GRADER_DEPLOYMENT_UPDATE:
        raise RuntimeError(
            "Review the grader deployment plan, then set "
            "FOUNDRY_ALLOW_GRADER_DEPLOYMENT_UPDATE=true to authorize the ARM update."
        )

    record_grader_deployment_operation(plan, existing)
    deployment = run_az_json(
        "cognitiveservices", "account", "deployment", "create",
        "--name", account_name,
        "--resource-group", RESOURCE_GROUP,
        "--subscription", SUBSCRIPTION_ID,
        "--deployment-name", GRADER_DEPLOYMENT,
        "--model-name", GRADER_MODEL,
        "--model-version", GRADER_MODEL_VERSION,
        "--model-format", "OpenAI",
        "--sku-name", GRADER_DEPLOYMENT_SKU,
        "--sku-capacity", str(plan["target_capacity"]),
    )
    validate_existing_grader(deployment)
    if int(deployment.get("sku", {}).get("capacity") or 0) != plan["target_capacity"]:
        raise RuntimeError(
            "ARM did not report the first requested grader capacity. "
            "Reconcile the update; the original cleanup baseline was preserved."
        )
    token_limit = deployment_token_limit(deployment)
    if token_limit < GRADER_MIN_TPM:
        raise RuntimeError(
            f"Grader deployment reports {token_limit:,} TPM after the update; "
            f"at least {GRADER_MIN_TPM:,} TPM is required. "
            "The original cleanup baseline is saved; reconcile before another update."
        )

    print(
        f"Grader deployment ready: capacity {plan['target_capacity']} "
        f"({token_limit:,} TPM). Original cleanup baseline retained."
    )
    return deployment


credential = project_client = openai_client = None
if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in .env or the process environment.")
grader_deployment = ensure_grader_deployment()
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)
openai_client = project_client.get_openai_client()

In [ ]:
def wait_for_file(file_id, timeout_seconds=600):
    deadline = time.monotonic() + timeout_seconds
    delay = 1
    while True:
        try:
            uploaded = openai_client.files.retrieve(file_id)
        except Exception as exc:
            if getattr(exc, "status_code", None) != 404 or time.monotonic() >= deadline:
                raise
            print(f"File {file_id} is propagating; retrying in {delay}s")
            time.sleep(delay)
            delay = min(delay * 2, 10)
            continue
        status = str(uploaded.status).casefold()
        if status in {"processed", "completed"}:
            return uploaded
        if status in {"error", "failed", "expired", "cancelled"}:
            raise RuntimeError(f"File {file_id} ended as {status}: {uploaded}")
        if time.monotonic() >= deadline:
            raise TimeoutError(f"File {file_id} exceeded {timeout_seconds} seconds")
        time.sleep(5)


def upload_or_reuse_exact(label):
    contract = DATASETS[label]
    content = contract["path"].read_bytes().replace(b"\r\n", b"\n")
    remote_name = (
        f"countdown-{label}-{contract['rows']}-"
        f"{contract['sha256'][:12]}.jsonl"
    )
    for remote_file in openai_client.files.list(limit=100):
        if remote_file.filename != remote_name:
            continue
        try:
            wait_for_file(remote_file.id, timeout_seconds=15)
            remote_content = openai_client.files.content(remote_file.id).read()
        except Exception as exc:
            if getattr(exc, "status_code", None) == 404 or isinstance(exc, TimeoutError):
                print("Ignoring orphaned file listing:", remote_file.filename, remote_file.id)
                continue
            raise
        remote_sha256 = hashlib.sha256(remote_content).hexdigest()
        if remote_sha256 != contract["sha256"]:
            raise RuntimeError(f"Remote file {remote_name} has unexpected bytes")
        print("Reusing exact file:", remote_file.filename, remote_file.id)
        return remote_file.id
    uploaded = openai_client.files.create(
        file=(remote_name, io.BytesIO(content), "application/jsonl"),
        purpose="fine-tune",
    )
    wait_for_file(uploaded.id)
    print("Uploaded:", remote_name, uploaded.id)
    return uploaded.id


training_file_id = validation_file_id = None
training_file_id = upload_or_reuse_exact("training")
validation_file_id = upload_or_reuse_exact("validation")


## 5. Define the training recipe and resumable submission

Both variants use evaluation every 5 steps with 2 samples, medium reasoning effort, 1 epoch, batch size 4, and learning-rate multiplier 2.

In [ ]:
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}


@contextmanager
def submission_state_lock():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    lock_path = STATE_PATH.with_name(STATE_PATH.name + ".lock")
    try:
        handle = lock_path.open("x", encoding="utf-8")
    except FileExistsError as exc:
        raise RuntimeError(
            "Submission state is locked by another writer or an interrupted process. "
            "Do not break a live lock. For a stale lock, first stop all notebook writers "
            "and independently reconcile pending/jobs; remove only the stale lock, "
            "never pending evidence. " + submission_reconciliation_message()
        ) from exc
    try:
        with handle:
            handle.write(json.dumps({
                "pid": os.getpid(),
                "recorded_at": datetime.now(timezone.utc).isoformat(),
                "project_endpoint": PROJECT_ENDPOINT,
            }, sort_keys=True) + "\n")
            handle.flush()
            os.fsync(handle.fileno())
            yield
    finally:
        lock_path.unlink()


def save_state(state):
    # Caller holds submission_state_lock across load, create/retrieve, and both saves.
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    content = json.dumps(state, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="\n",
            prefix=STATE_PATH.name + ".", suffix=".tmp",
            dir=STATE_PATH.parent, delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            handle.write(content)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary_path, STATE_PATH)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def submit_once(label, grader, suffix):
    payload = build_job_payload(
        label, grader, suffix, training_file_id, validation_file_id,
    )
    if recipe_fingerprint() != RUN_FINGERPRINT:
        raise RuntimeError("The recipe changed after validation; rerun the offline preparation cells.")
    with submission_state_lock():
        state = load_state()
        existing_job_id = state["jobs"].get(label)
        if existing_job_id:
            job = openai_client.fine_tuning.jobs.retrieve(existing_job_id)
            print(f"Resuming {label}: {job.id} ({job.status})")
            return job
        if label in state["pending"]:
            raise RuntimeError(
                "A pending submission has no verified job ID; no POST was replayed. "
                + submission_reconciliation_message(label)
            )

        intent = build_submission_intent(label, payload)
        state["fingerprint"] = RUN_FINGERPRINT
        state["pending"][label] = intent
        # No network call is allowed until the exact intent has been durably replaced.
        try:
            save_state(state)
            persisted = load_state()
            if persisted != state:
                raise RuntimeError("Submission intent was not persisted exactly; no POST was sent.")
        except Exception as exc:
            raise RuntimeError(
                "Saving or verifying the pre-POST submission intent failed; no POST was sent. "
                "Any existing pending evidence was retained. "
                + submission_reconciliation_message(label)
            ) from exc
        try:
            # The actual SDK payload is detached from immutable pending evidence.
            request_payload = json.loads(json.dumps(intent["payload"], allow_nan=False))
            job = openai_client.with_options(max_retries=0).fine_tuning.jobs.create(**request_payload)
        except Exception as exc:
            # Even apparently definitive errors cannot authorize replay of a paid POST.
            raise RuntimeError(
                "Job creation failed or its response is ambiguous; the pending intent was retained. "
                + submission_reconciliation_message(label)
            ) from exc
        job_id = getattr(job, "id", None)
        if not isinstance(job_id, str) or not job_id or job_id != job_id.strip():
            raise RuntimeError(
                "Job creation did not return a usable job ID; the pending intent was retained. "
                + submission_reconciliation_message(label)
            )
        state["jobs"][label] = job_id
        try:
            # Keep the original per-label intent even after a known ID is recorded.
            save_state(state)
        except Exception as exc:
            raise RuntimeError(
                "The service returned a job ID but saving it failed; do not submit again. "
                + submission_reconciliation_message(label, job_id)
            ) from exc
        print(f"Submitted {label}: {job.id}")
        return job


def wait_for_terminal(label, job_id, poll_seconds=60, timeout_seconds=21_600):
    previous_status = None
    deadline = time.monotonic() + timeout_seconds
    while True:
        if time.monotonic() >= deadline:
            raise TimeoutError(f"{label} exceeded {timeout_seconds} seconds; rerun to resume.")
        try:
            job = openai_client.fine_tuning.jobs.retrieve(job_id)
        except Exception as exc:
            status_code = getattr(exc, "status_code", None)
            retryable = (
                status_code == 429
                or (status_code is not None and status_code >= 500)
                or type(exc).__name__
                in {
                    "APIConnectionError",
                    "APITimeoutError",
                    "ConnectError",
                    "ConnectTimeout",
                    "CredentialUnavailableError",
                    "ReadTimeout",
                    "TimeoutExpired",
                }
            )
            if not retryable:
                raise
            print(f"{label}: transient polling error; retrying: {exc}")
            time.sleep(poll_seconds)
            continue
        if job.status != previous_status:
            print(
                f"[{datetime.now(timezone.utc).isoformat()}] "
                f"{label}: {job.id} -> {job.status}"
            )
            previous_status = job.status
        if job.status in TERMINAL_STATUSES:
            if job.status != "succeeded":
                raise RuntimeError(f"{label} ended as {job.status}: {job.error}")
            return job
        time.sleep(poll_seconds)


def download_results_and_summarize(label, job):
    result_root = OUTPUT_ROOT / label
    result_root.mkdir(parents=True, exist_ok=True)
    rows = []
    files = []
    for index, file_id in enumerate(job.result_files or [], start=1):
        content = openai_client.files.content(file_id).read()
        path = result_root / f"result-{index}.csv"
        path.write_bytes(content)
        parsed = list(csv.DictReader(io.StringIO(content.decode("utf-8-sig"))))
        rows.extend(parsed)
        files.append(
            {
                "file_id": file_id,
                "path": str(path.relative_to(DEMO_ROOT)).replace("\\", "/"),
                "bytes": len(content),
                "sha256": hashlib.sha256(content).hexdigest(),
                "rows": len(parsed),
            }
        )

    validation_columns = [
        column
        for column in (rows[0].keys() if rows else [])
        if "reward" in column.lower()
        and ("valid" in column.lower() or "validation" in column.lower())
        and "error" not in column.lower()
    ]
    curve = []
    for row in rows:
        for column in validation_columns:
            value = row.get(column)
            if value not in {None, ""}:
                curve.append(
                    {
                        "step": int(float(row["step"])) if row.get("step") else None,
                        "column": column,
                        "value": float(value),
                    }
                )
                break
    values = [point["value"] for point in curve]
    if not values:
        raise RuntimeError(f"{label}: no service validation reward metrics returned")
    summary = {
        "label": label,
        "job_id": job.id,
        "status": job.status,
        "model": job.model,
        "training_type": TRAINING_TYPE,
        "trained_tokens": job.trained_tokens,
        "error": job.error.model_dump(mode="json") if job.error else None,
        "input_hashes": {
            key: DATASETS[key]["sha256"]
            for key in ("training", "validation", "evaluation_provenance")
        },
        "input_evidence": {
            key: {
                "raw_bytes": len(DATASETS[key]["path"].read_bytes()),
                "raw_sha256": hashlib.sha256(
                    DATASETS[key]["path"].read_bytes()
                ).hexdigest(),
                "normalized_bytes": len(
                    DATASETS[key]["path"].read_bytes().replace(b"\r\n", b"\n")
                ),
                "normalized_sha256": DATASETS[key]["sha256"],
            }
            for key in ("training", "validation", "evaluation_provenance")
        },
        "result_files": files,
        "validation_curve": curve,
        "validation_reward_initial": values[0] if values else None,
        "validation_reward_final": values[-1] if values else None,
        "validation_reward_max": max(values) if values else None,
    }
    (result_root / "summary.json").write_text(
        json.dumps(summary, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(summary, indent=2, default=str))
    return summary

## 6. Submit or resume both jobs

Before each POST, an exclusive local state lock protects a flushed/fsynced, atomically replaced per-label pending intent containing the recipe identity, exact payload and payload hash, and exact uploaded file IDs with canonical dataset hashes. SDK retries are disabled. Known job IDs resume through retrieval; the original intent remains saved even after success. A save failure before POST prevents creation; a save failure after a returned ID reports that ID while retaining pending evidence.

A pending intent without a known ID blocks all automatic replay for that label, even after a fresh kernel or an apparently definitive error. Preserve the original state bytes privately. Never clear, delete, or archive pending to resubmit. Independently verify the project and match the saved endpoint, payload/suffix, file IDs/hashes, and service request/audit evidence. Only after proving a matching job may an operator record its verified ID in `jobs[label]`, retaining `pending[label]` with all writers stopped; rerunning then retrieves it. If no matching ID can be proved, stop and contact the service operator. Absence from a job listing is not proof that no job was accepted.

Do not break a live state lock. Stop all writers and reconcile state before removing only a confirmed stale lock after interruption. The two labels keep independent intents; this local safeguard is not a distributed or service-side idempotency guarantee. No live deployment or training success is claimed by these checks.

In [ ]:
model_grader_job = submit_once(
    label="model-grader",
    grader=MODEL_GRADER,
    suffix="qwen36-countdown-model-grader",
)
python_grader_job = submit_once(
    label="python-grader",
    grader=PYTHON_GRADER,
    suffix="qwen36-countdown-python-grader",
)


## 7. Monitor the model/score-grader job and download its result CSV

In [ ]:
model_grader_job = wait_for_terminal("model-grader", model_grader_job.id)
model_grader_summary = download_results_and_summarize(
    "model-grader",
    model_grader_job,
)


## 8. Monitor the deterministic Python-grader job and download its result CSV

In [ ]:
python_grader_job = wait_for_terminal("python-grader", python_grader_job.id)
python_grader_summary = download_results_and_summarize(
    "python-grader",
    python_grader_job,
)


## 9. Reward-curve summary

In [ ]:
reward_summary = {
    "model-grader": {
        "status": model_grader_summary["status"],
        "initial": model_grader_summary["validation_reward_initial"],
        "final": model_grader_summary["validation_reward_final"],
        "max": model_grader_summary["validation_reward_max"],
    },
    "python-grader": {
        "status": python_grader_summary["status"],
        "initial": python_grader_summary["validation_reward_initial"],
        "final": python_grader_summary["validation_reward_final"],
        "max": python_grader_summary["validation_reward_max"],
    },
}
print(json.dumps(reward_summary, indent=2))
openai_client.close()
project_client.close()
credential.close()


### Interpret the training metrics

Use the generated `reward_summary` and the saved result CSVs to inspect each grader's validation-reward trajectory. These values are service training signals for this run; they do not establish held-out task quality or deployment behavior.

The grader deployment is not automatically resized or deleted. Keep `outputs/grader-deployment-state.json` even when deployment or submission failed: it preserves the first pre-update cleanup baseline, not proof of successful creation or exclusive ownership. Reconcile uncertain operations and all dependent jobs, verify ownership independently, and only after the relevant jobs are terminal deliberately restore the original capacity or remove an owned deployment through your approved resource-management process.